# LoRA fine-tune: opus-mt-en-ga

Runs on Colab free T4. Upload `train.tsv` and `dev.tsv` (from `scripts/prepare_data.py`) into the Colab session before running cell 3.

Output: a LoRA adapter saved to `./adapter/` and zipped for download.

In [ ]:
!pip -q install transformers==4.44.2 peft==0.13.0 accelerate==0.34.2 datasets==2.21.0 sentencepiece sacremoses sacrebleu

In [ ]:
from google.colab import files
uploaded = files.upload()  # upload train.tsv and dev.tsv

In [ ]:
from datasets import load_dataset
raw = load_dataset('csv', data_files={'train': 'train.tsv', 'dev': 'dev.tsv'}, delimiter='\t')
print(raw)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
MODEL_ID = 'Helsinki-NLP/opus-mt-en-ga'
tok = AutoTokenizer.from_pretrained(MODEL_ID)
base = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID)

MAX = 128
def preprocess(batch):
    src = [s or '' for s in batch['src']]
    tgt = [t or '' for t in batch['tgt']]
    enc = tok(src, max_length=MAX, truncation=True)
    with tok.as_target_tokenizer():
        lab = tok(tgt, max_length=MAX, truncation=True)
    enc['labels'] = lab['input_ids']
    return enc

ds = raw.map(preprocess, batched=True, remove_columns=raw['train'].column_names)
print(ds)

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType
lora = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=['q_proj','k_proj','v_proj','out_proj'],
)
model = get_peft_model(base, lora)
model.print_trainable_parameters()

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
collator = DataCollatorForSeq2Seq(tok, model=model)
args = Seq2SeqTrainingArguments(
    output_dir='out',
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=3e-4,
    warmup_ratio=0.05,
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_steps=50,
    predict_with_generate=False,
    fp16=True,
    report_to='none',
)
trainer = Seq2SeqTrainer(
    model=model, args=args,
    train_dataset=ds['train'], eval_dataset=ds['dev'],
    tokenizer=tok, data_collator=collator,
)
trainer.train()

In [ ]:
model.save_pretrained('adapter')
tok.save_pretrained('adapter')
!zip -qr adapter.zip adapter
from google.colab import files
files.download('adapter.zip')